#**Cold Storage Alarm Detection Algorithms**

*   **Objective:** Simulate real-time temperature data from multiple cold storage shelves and implement algorithms to detect quality-critical thermal excursions.
*   **Prerequisite Knowledge:** Basic Python, NumPy (for data generation), and statistical concepts.



In [ ]:
import numpy as np
import random

1. Configuration and Simulation Parameters

In [ ]:
# Defined temperature requirement for sensitive pharmaceuticals (e.g., vaccines)
REQUIRED_TEMP_MIN = 2.0  # Celsius
REQUIRED_TEMP_MAX = 8.0  # Celsius

# Total number of storage shelves/locations being monitored
NUM_SHELVES = 10

# Simulated data points (time steps)
SIMULATION_STEPS = 15

# Configuration for introducing an intentional excursion (for testing)
EXCURSION_SHELF_INDEX = 5  # Shelf 6 (index 5)
EXCURSION_START_STEP = 8    # Step 9

2. Data Generation Function
    * Generates simulated temperature data for all shelves over multiple time steps.
    * Introduces a controlled thermal excursion for testing algorithm robustness.

In [ ]:
def generate_simulated_data(steps, num_shelves, excursion_shelf, excursion_start):

    print(f"--- Simulating Data for {num_shelves} Shelves over {steps} Steps ---")
    print(f"Target Range: [{REQUIRED_TEMP_MIN}°C, {REQUIRED_TEMP_MAX}°C]")
    print(f"Injecting Excursion: Shelf {excursion_shelf + 1} starting at Step {excursion_start + 1}\n")

    # Initial data centered around the ideal 5.0 C, with minor variance (0.5 C std dev)
    data = np.random.normal(loc=5.0, scale=0.5, size=(steps, num_shelves))

    # --- Inject the Thermal Excursion (Simulating a Door Left Open) ---
    for t in range(excursion_start, steps):
        # Temperatures for the compromised shelf start rising significantly
        # The rising factor simulates the ambient air gradually increasing the temperature.
        rising_factor = (t - excursion_start + 1) * 0.75
        data[t, excursion_shelf] += rising_factor
        # Add a bit of natural noise to the excursion
        data[t, excursion_shelf] += random.uniform(-0.5, 0.5)

    # Round the data for cleaner output
    data = np.round(data, 2)
    return data

3. Alarm Detection Algorithms


*   Algorithm 1: Detects alarm if temperature exceeds the fixed min/max thresholds. This is the most common and critical alarm type.
    






In [ ]:
def detect_hard_threshold_alarm(temp_now, shelf_index):

    shelf_name = f"Shelf {shelf_index + 1}"

    if temp_now < REQUIRED_TEMP_MIN:
        print(f"[{shelf_name}] ALARM 1 (Threshold): LOW - {temp_now}°C (Below {REQUIRED_TEMP_MIN}°C)")
        return True, "LOW"
    elif temp_now > REQUIRED_TEMP_MAX:
        print(f"[{shelf_name}] ALARM 1 (Threshold): HIGH - {temp_now}°C (Above {REQUIRED_TEMP_MAX}°C)")
        return True, "HIGH"
    return False, None

* Algorithm 2: Detects alarm if a single shelf's temperature is statistically out of sync with the average of *all other shelves* at the current time step. This detects isolated sensor drift or localized hot/cold spots.




In [ ]:
def detect_statistical_alarm(temperatures_all, temp_now, shelf_index, threshold_std_dev=3.0):

    shelf_name = f"Shelf {shelf_index + 1}"

    # Calculate the mean and standard deviation of *all* shelves at this time step
    mean_all = np.mean(temperatures_all)
    std_all = np.std(temperatures_all)

    # Check if the temperature is more than 'threshold_std_dev' standard deviations away from the mean
    if std_all > 0 and abs(temp_now - mean_all) > threshold_std_dev * std_all:
        print(f"[{shelf_name}] ALARM 2 (Statistical): Deviation - {temp_now}°C | Avg: {mean_all:.2f}°C | StdDev: {std_all:.2f}°C")
        return True, "STATISTICAL"
    return False, None

 * Algorithm 3: Triggers if the temperature of a single shelf has increased by more than 2.0°C between the current step (t) and the previous step (t-1).

In [ ]:
def detect_rate_of_change_alarm(temp_now, step, shelf_index, all_data):

    if step == 0:
        return False, None # Cannot calculate rate of change on the first step

    # Get the temperature from the previous step (t-1) for the same shelf
    temp_prev = all_data[step - 1, shelf_index]

    # Calculate the rate of change
    rate_of_change = temp_now - temp_prev

    # Check if the increase exceeds the threshold
    RATE_THRESHOLD = 2.0
    if rate_of_change > RATE_THRESHOLD:
        shelf_name = f"Shelf {shelf_index + 1}"
        print(f"[{shelf_name}] ALARM 3 (Rate of Change): +{rate_of_change:.2f}°C / Step | Prev: {temp_prev}°C | Current: {temp_now}°C")
        return True, "RATE_HIGH"

    return False, None


4. Main Simulation Loop
   * Executes the entire simulation, generating data and running all algorithms for every shelf at every time step.

In [ ]:
def run_simulation():

    # Generate the full dataset
    data = generate_simulated_data(SIMULATION_STEPS, NUM_SHELVES, EXCURSION_SHELF_INDEX, EXCURSION_START_STEP)

    total_alarms = 0
    print("\n" + "="*80)
    print("STARTING REAL-TIME MONITORING AND ALGORITHM EXECUTION")
    print("="*80)

    # Iterate through each time step (T)
    for t in range(SIMULATION_STEPS):
        print(f"\n--- STEP {t + 1} / {SIMULATION_STEPS} (Time) ---")
        current_temperatures = data[t, :]
        step_alarms = 0

        # Iterate through each shelf (S) at the current time step
        for s in range(NUM_SHELVES):
            temp_now = current_temperatures[s]
            shelf_name = f"Shelf {s + 1}"

            # Priority 1: Threshold Check (The most critical failure)
            is_alarm1, severity1 = detect_hard_threshold_alarm(temp_now, s)
            if is_alarm1:
                total_alarms += 1
                step_alarms += 1
                continue # Skip other checks if a hard failure occurred

            # Priority 2: Rate of Change Check (Early warning for rapid failure)
            is_alarm3, severity3 = detect_rate_of_change_alarm(temp_now, t, s, data)
            if is_alarm3:
                total_alarms += 1
                step_alarms += 1
                continue # Skip Statistical check if Rate of Change detected a problem

            # Priority 3: Statistical Check (Early warning for isolated drift)
            is_alarm2, severity2 = detect_statistical_alarm(current_temperatures, temp_now, s)
            if is_alarm2:
                total_alarms += 1
                step_alarms += 1

            # Print current non-alarming status
            if not is_alarm1 and not is_alarm2 and not is_alarm3:
                print(f"[{shelf_name}] OK: {temp_now}°C")

    print("\n" + "="*80)
    print(f"SIMULATION COMPLETE. Total Alarms Detected: {total_alarms}")
    print("="*80)
    print("\nLAB TASKS:")
    print("1. Locate the time step and shelf index where the THRESHOLD alarm (Alarm 1) first triggered.")
    print("2. Change the 'threshold_std_dev' in Algorithm 2 (Statistical) to 1.0 (inside the function definition). How does this impact the detection?")
    print("3. TASK: Analyze the Rate of Change Alarm (Alarm 3) you implemented. When did it trigger relative to the Hard Threshold Alarm (Alarm 1)?")
    print("4. Discuss: Which alarm type (Threshold, Statistical, or Rate of Change) is more critical for ISO 13485 compliance, and why?")



In [ ]:
if __name__ == "__main__":
    # Ensure numpy is installed: pip install numpy
    run_simulation()

--- Simulating Data for 10 Shelves over 15 Steps ---
Target Range: [2.0°C, 8.0°C]
Injecting Excursion: Shelf 6 starting at Step 9


STARTING REAL-TIME MONITORING AND ALGORITHM EXECUTION

--- STEP 1 / 15 (Time) ---
[Shelf 1] OK: 5.0°C
[Shelf 2] OK: 4.67°C
[Shelf 3] OK: 4.84°C
[Shelf 4] OK: 4.83°C
[Shelf 5] OK: 4.91°C
[Shelf 6] OK: 4.56°C
[Shelf 7] OK: 5.91°C
[Shelf 8] OK: 4.55°C
[Shelf 9] OK: 5.88°C
[Shelf 10] OK: 4.51°C

--- STEP 2 / 15 (Time) ---
[Shelf 1] OK: 4.7°C
[Shelf 2] OK: 5.15°C
[Shelf 3] OK: 5.1°C
[Shelf 4] OK: 4.61°C
[Shelf 5] OK: 4.02°C
[Shelf 6] OK: 5.81°C
[Shelf 7] OK: 4.55°C
[Shelf 8] OK: 5.44°C
[Shelf 9] OK: 5.27°C
[Shelf 10] OK: 5.59°C

--- STEP 3 / 15 (Time) ---
[Shelf 1] OK: 5.24°C
[Shelf 2] OK: 6.26°C
[Shelf 3] OK: 4.87°C
[Shelf 4] OK: 5.09°C
[Shelf 5] OK: 5.26°C
[Shelf 6] OK: 4.88°C
[Shelf 7] OK: 4.83°C
[Shelf 8] OK: 4.06°C
[Shelf 9] OK: 5.23°C
[Shelf 10] OK: 4.36°C

--- STEP 4 / 15 (Time) ---
[Shelf 1] OK: 4.69°C
[Shelf 2] OK: 5.31°C
[Shelf 3] OK: 5.96°C
[Shelf 4]